In [1]:
import tensorflow as tf
import os
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import json

"""
PHYSICS-CONSTRAINED OPEN-SET LIVESTOCK IDENTIFICATION

Final Objective Function (6 Components):
    L = L_arc + λ1·L_center + λ2·L_unknown + λ3·L_sym + λ4·L_energy + λ5·L_lap

Ablation Configurations:
    L1 — ArcFace only
        CENTER=0.0, UNKNOWN=0.0, SYM=0.0, LAP=0.0, ENERGY=0.0

    L2 — ArcFace + Unknown Rejection
        CENTER=0.0, UNKNOWN=0.3, SYM=0.0, LAP=0.0, ENERGY=0.0

    L3 — ArcFace + Unknown + Symmetry + Smoothness
        CENTER=0.0, UNKNOWN=0.3, SYM=0.01, LAP=0.01, ENERGY=0.0

    L4 — ArcFace + Unknown + Symmetry + Smoothness + Energy
        CENTER=0.0, UNKNOWN=0.3, SYM=0.01, LAP=0.01, ENERGY=0.05
"""


# SEEDS & GPU

np.random.seed(42)
tf.random.set_seed(42)

gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass


# CONFIGURATION  

dataset_root = "/home/student/Desktop/Kumar/dataset"

class CFG:
    # Model
    IMG_SIZE       = 224
    EMBED_DIM      = 512
    MARGIN         = 0.5
    SCALE          = 64
    BATCH_SIZE     = 32


    MODEL_TAG      = "L3"   

    # Loss weights
    CENTER_LOSS_WEIGHT   = 0.0   # λ1 — center loss
    UNKNOWN_LOSS_WEIGHT  = 0.3   # λ2 — unknown rejection    (L2+: 0.3)
    PHYSICS_SYM_WEIGHT   = 0.05   # λ3 — symmetry             (L3+: 0.01)
    PHYSICS_LAP_WEIGHT   = 0.05   # λ5 — smoothness           (L3+: 0.01)
    PHYSICS_ENERGY_WEIGHT= 0.0   # λ4 — energy margin        (L4:  0.05)

    # Physics schedule 
    PHYSICS_WARMUP_EPOCHS = 10   # ramp physics weights from 0 → target

    # Energy
    ENERGY_TAU    = 0.1          # LogSumExp temperature
    ENERGY_MARGIN = 0.2          # per-sample margin

    # Training schedule
    STEPS_PER_EPOCH = 200
    AUTOTUNE        = tf.data.AUTOTUNE
    TRAIN_SPLIT     = 0.85       # per-cattle train/val split

    PHASE_1_EPOCHS  = 5          # 100% known
    PHASE_2_EPOCHS  = 15         # 80% known / 20% unknown
    PHASE_3_EPOCHS  = 10         # 60% known / 40% unknown
    TOTAL_EPOCHS    = PHASE_1_EPOCHS + PHASE_2_EPOCHS + PHASE_3_EPOCHS

    # Output paths
    @classmethod
    def ckpt(cls, phase): return f"cow_{cls.MODEL_TAG}_phase{phase}.weights.h5"
    @classmethod
    def saved_model_dir(cls): return f"cow_{cls.MODEL_TAG}_saved_model"
    @classmethod
    def gallery_path(cls): return f"cow_{cls.MODEL_TAG}_train_gallery.npz"
    @classmethod
    def centers_path(cls): return f"cow_{cls.MODEL_TAG}_class_centers.npy"
    @classmethod
    def metadata_path(cls): return f"cow_{cls.MODEL_TAG}_metadata.json"


print(f"{'='*70}")
print(f"  TRAINING  {CFG.MODEL_TAG}")
print(f"  Weights → unknown={CFG.UNKNOWN_LOSS_WEIGHT}  "
      f"sym={CFG.PHYSICS_SYM_WEIGHT}  "
      f"lap={CFG.PHYSICS_LAP_WEIGHT}  "
      f"energy={CFG.PHYSICS_ENERGY_WEIGHT}")
print(f"{'='*70}")



# DATA LOADING


def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

def augment_image(img):
    """Standard augmentation for known samples."""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.25)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    return img

def augment_unknown(img):
    """Heavy augmentation to create synthetic unknowns."""
    img = tf.image.random_brightness(img, 0.5)
    img = tf.image.random_contrast(img, 0.5, 1.5)
    img = tf.image.random_saturation(img, 0.5, 1.5)
    img = tf.image.random_hue(img, 0.2)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=0.1)
    img = img + noise
    img = tf.clip_by_value(img, -2.0, 2.0)
    return img


class CombinedDatasetBuilder:
    """
    Combines cattle images from face_train / face_test / face_val folders,
    then splits each cattle's images into train and val portions.

    After training you can call get_all_paths_and_labels() to extract
    a full embedding gallery for genuine-score evaluation.
    """
    def __init__(self, face_folders, split='train', train_split=0.85, seed=42):
        self.split      = split
        self.cow_data   = {}          # cow_id → [all image paths]
        self.split_data = {}          # cow_id → [split image paths]

        np.random.seed(seed)

        # Gather all images per cattle 
        for folder in face_folders:
            if not os.path.exists(folder):
                print(f"  Warning: {folder} not found, skipping.")
                continue
            for cls in sorted(os.listdir(folder)):
                cls_path = os.path.join(folder, cls)
                if not os.path.isdir(cls_path):
                    continue
                imgs = [os.path.join(cls_path, f)
                        for f in os.listdir(cls_path)
                        if f.lower().endswith((".jpg", ".jpeg", ".png"))]
                if imgs:
                    self.cow_data.setdefault(cls, []).extend(imgs)

        # Per-cattle train / val split 
        for cow_id, img_list in self.cow_data.items():
            img_list = sorted(img_list)
            idx = np.arange(len(img_list))
            np.random.shuffle(idx)
            img_list = [img_list[i] for i in idx]

            n_train = int(len(img_list) * train_split)
            self.split_data[cow_id] = (
                img_list[:n_train] if split == 'train' else img_list[n_train:]
            )

        self.classes     = sorted(self.split_data.keys())
        self.cls_to_idx  = {c: i for i, c in enumerate(self.classes)}
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.split_data.values())

    # tf.data pipeline 
    def get_tfdata_unbatched(self, shuffle=True, augment=False):
        paths, labels = [], []
        for cow_id, img_list in self.split_data.items():
            for p in img_list:
                paths.append(p)
                labels.append(self.cls_to_idx[cow_id])

        ds = tf.data.Dataset.from_tensor_slices(
            {'face_path': paths, 'label': labels}
        )

        def map_fn(sample):
            img = load_image(sample['face_path'])
            if augment:
                img = augment_image(img)
            return {'face_input': img, 'label_input': sample['label']}, sample['label']

        ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
        if shuffle:
            ds = ds.shuffle(2000, seed=42, reshuffle_each_iteration=True)
        return ds.repeat()

    # Gallery extraction helper (used AFTER training) 
    def get_all_paths_and_labels(self):
        """
        Returns (paths, integer_labels, class_names).
        Used post-training to build the training gallery for genuine-score
        evaluation: pass each path through the inference model and store
        the embedding alongside its label.
        """
        paths, labels = [], []
        for cow_id, img_list in self.split_data.items():
            for p in img_list:
                paths.append(p)
                labels.append(self.cls_to_idx[cow_id])
        return paths, labels, self.classes


# Load datasets
print("\nLOADING DATA")

all_face_folders = [
    os.path.join(dataset_root, "face_train"),
    os.path.join(dataset_root, "face_test"),
    os.path.join(dataset_root, "face_val"),
]

train_data = CombinedDatasetBuilder(all_face_folders, split='train',
                                    train_split=CFG.TRAIN_SPLIT)
val_data   = CombinedDatasetBuilder(all_face_folders, split='val',
                                    train_split=CFG.TRAIN_SPLIT)

print(f"\n✓ {train_data.num_classes} cattle loaded")
print(f"  Train : {train_data.total_images} images ({CFG.TRAIN_SPLIT*100:.0f}%)")
print(f"  Val   : {val_data.total_images}   images ({(1-CFG.TRAIN_SPLIT)*100:.0f}%)")


# MODEL ARCHITECTURE


def build_model(num_classes, embedding_size=512, margin=0.5, scale=64):
    """
    EfficientNetB0 → Dense(512) [unnorm] → L2-Normalize → ArcFace

    Outputs: (logits, embeddings_norm, embeddings_unnorm)
    The unnormalized path is kept for energy margin loss.
    """
    face_input  = keras.Input(shape=(224, 224, 3), name="face_input")
    label_input = keras.Input(shape=(), dtype=tf.int32, name="label_input")

    # Backbone
    backbone = keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights="imagenet"
    )
    # Freeze all but the last 30 layers; keep BN frozen throughout
    for layer in backbone.layers[:-30]:
        layer.trainable = False
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    x = backbone(face_input, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    # Embedding head 
    x_unnorm = layers.Dense(
        embedding_size,
        kernel_regularizer=keras.regularizers.l2(1e-4),
        name="embedding_dense"
    )(x)

    embeddings = layers.Lambda(
        lambda z: tf.math.l2_normalize(z, axis=1),
        name="l2_normalize"
    )(x_unnorm)

    # ArcFace layer
    class SafeArcFaceLayer(layers.Layer):
        def __init__(self, n_classes, margin=0.5, scale=64, **kwargs):
            super().__init__(**kwargs)
            self.n_classes = n_classes
            self.margin    = margin
            self.scale     = scale
            self.cos_m     = tf.cos(margin)
            self.sin_m     = tf.sin(margin)

        def build(self, input_shape):
            self.W = self.add_weight(
                name="arcface_weights",
                shape=(input_shape[0][-1], self.n_classes),
                initializer=keras.initializers.HeNormal(),
                trainable=True,
                regularizer=keras.regularizers.l2(1e-4)
            )
            super().build(input_shape)

        def call(self, inputs):
            emb, labels = inputs
            W_norm   = tf.nn.l2_normalize(self.W, axis=0)
            emb_norm = tf.nn.l2_normalize(emb, axis=1)

            cos_theta = tf.clip_by_value(
                tf.matmul(emb_norm, W_norm), -1.0 + 1e-7, 1.0 - 1e-7
            )
            sin_theta     = tf.sqrt(tf.maximum(1.0 - tf.square(cos_theta), 1e-7))
            target_logits = cos_theta * self.cos_m - sin_theta * self.sin_m

            labels      = tf.cast(labels, tf.int32)
            known_mask  = tf.greater_equal(labels, 0)
            labels_safe = tf.where(known_mask, labels, tf.zeros_like(labels))

            onehot = (tf.one_hot(labels_safe, depth=self.n_classes, dtype=tf.float32)
                      * tf.expand_dims(tf.cast(known_mask, tf.float32), axis=1))

            output = cos_theta * (1.0 - onehot) + target_logits * onehot
            return output * self.scale

        def get_config(self):
            return {"n_classes": self.n_classes, "margin": self.margin,
                    "scale": self.scale}

    logits = SafeArcFaceLayer(
        num_classes, margin=margin, scale=scale, name="arcface"
    )([embeddings, label_input])

    return keras.Model(
        inputs=[face_input, label_input],
        outputs=[logits, embeddings, x_unnorm],
        name=f"cow_full_training_{CFG.MODEL_TAG}"
    )


print("\nBUILDING MODEL")
model = build_model(train_data.num_classes, CFG.EMBED_DIM, CFG.MARGIN, CFG.SCALE)

# Embedding-only sub-model (image input → norm + unnorm embeddings)
embedding_model = keras.Model(
    inputs=model.input[0],
    outputs=[model.output[1], model.output[2]],
    name="embedding_extractor"
)
print(f"  ✓ Model built: EfficientNetB0 → Dense(512) → L2-Norm → ArcFace")
print(f"  ✓ Embedding extractor ready")



# LOSS LAYERS


class CenterLossLayer(layers.Layer):
    """
    Center loss — pulls each embedding toward its class center.
    Centers are updated with exponential moving average (not via backprop).
    """
    def __init__(self, num_classes, embedding_dim, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.num_classes   = num_classes
        self.embedding_dim = embedding_dim
        self.alpha         = tf.constant(alpha, dtype=tf.float32)

    def build(self, input_shape):
        self.centers = self.add_weight(
            name="class_centers",
            shape=(self.num_classes, self.embedding_dim),
            initializer="zeros",
            trainable=False
        )
        super().build(input_shape)

    def call(self, embeddings, labels, training=True):
        labels      = tf.cast(labels, tf.int32)
        known_mask  = labels >= 0
        known_emb   = tf.boolean_mask(embeddings, known_mask)
        known_lbls  = tf.boolean_mask(labels, known_mask)

        if tf.shape(known_emb)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)

        centers_batch = tf.gather(self.centers, known_lbls)
        diff  = known_emb - centers_batch
        loss  = tf.reduce_mean(tf.reduce_sum(tf.square(diff), axis=1))

        if training:
            unique_labels, idx = tf.unique(known_lbls)
            n = tf.shape(unique_labels)[0]
            summed = tf.math.unsorted_segment_sum(known_emb, idx, n)
            counts = tf.reshape(
                tf.math.unsorted_segment_sum(
                    tf.ones_like(idx, dtype=tf.float32), idx, n
                ), (-1, 1)
            )
            new_centers = summed / (counts + 1e-8)
            cur_centers = tf.gather(self.centers, unique_labels)
            updated     = self.alpha * cur_centers + (1.0 - self.alpha) * new_centers
            self.centers.assign(
                tf.tensor_scatter_nd_update(
                    self.centers, tf.expand_dims(unique_labels, 1), updated
                )
            )
        return loss


class UnknownRejectionLoss(layers.Layer):
    """
    Pushes unknown embeddings away from all known class prototypes.
    Minimises the maximum cosine similarity of each unknown to any class.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, embeddings, labels, class_weights):
        labels       = tf.cast(labels, tf.int32)
        unknown_mask = tf.less(labels, 0)
        unk_emb      = tf.boolean_mask(embeddings, unknown_mask)

        if tf.shape(unk_emb)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)

        W_norm       = tf.nn.l2_normalize(tf.stop_gradient(class_weights), axis=0)
        similarities = tf.matmul(unk_emb, W_norm)
        max_sims     = tf.reduce_max(similarities, axis=1)
        return tf.reduce_mean(max_sims)



# PHYSICS LOSSES


def get_physics_weights_tf(epoch, warmup_epochs=10):
    """Linear warmup from 0 → target weight over `warmup_epochs` epochs."""
    scale = tf.minimum(
        tf.cast(epoch, tf.float32) / tf.cast(warmup_epochs, tf.float32),
        1.0
    )
    return (
        CFG.PHYSICS_SYM_WEIGHT    * scale,
        CFG.PHYSICS_ENERGY_WEIGHT * scale,
        CFG.PHYSICS_LAP_WEIGHT    * scale,
    )


def compute_smooth_energy(embeddings_unnorm, class_weights, tau=0.1):
    """
    Free energy score computed on PRE-L2 embeddings.

    E(z) = -τ · log Σ_k exp(cos(z, w_k) / τ)

    Lower energy  → embedding is close to some known class (in-distribution).
    Higher energy → embedding is far from all classes (out-of-distribution).
    """
    W_frozen = tf.stop_gradient(class_weights)
    W_norm   = tf.nn.l2_normalize(W_frozen, axis=0)
    z_norm   = tf.nn.l2_normalize(embeddings_unnorm, axis=1)

    cos_sim  = tf.matmul(z_norm, W_norm)
    cos_sim  = cos_sim - tf.reduce_max(cos_sim, axis=1, keepdims=True)   # stability

    log_sum  = tf.math.log(tf.reduce_sum(tf.exp(cos_sim / tau), axis=1) + 1e-8)
    return -tau * log_sum


def energy_margin_loss(known_unnorm, unknown_unnorm, class_weights,
                       margin=0.2, tau=0.1):
    """
    Per-sample pairwise energy margin loss.

    Enforces: E(unknown_i) > E(known_j) + margin  for every (i, j) pair.
    Stronger than batch-mean comparison — prevents individual failures.
    """
    E_known   = compute_smooth_energy(known_unnorm,   class_weights, tau)
    E_unknown = compute_smooth_energy(unknown_unnorm, class_weights, tau)

    # (n_unknown, 1) - (1, n_known) → (n_unknown, n_known)
    pairwise  = tf.expand_dims(E_unknown, 1) - tf.expand_dims(E_known, 0)
    loss      = tf.reduce_mean(tf.maximum(0.0, margin - pairwise))

    return loss, tf.reduce_mean(E_known), tf.reduce_mean(E_unknown)



# TRAINER


center_loss_layer      = CenterLossLayer(train_data.num_classes, CFG.EMBED_DIM)
unknown_rejection_layer = UnknownRejectionLoss()


class PhysicsTrainer(keras.Model):
    """
    Custom training loop that orchestrates all 6 loss components.
    The only things that change between ablations are the weight constants
    in CFG — this class does not need to be modified.
    """

    def __init__(self, base_model, embedding_model,
                 center_loss_layer, unknown_rejection_layer,
                 center_loss_weight, unknown_loss_weight,
                 physics_warmup_epochs, **kwargs):
        super().__init__(**kwargs)
        self.base_model              = base_model
        self.embedding_model         = embedding_model
        self.center_loss_layer       = center_loss_layer
        self.unknown_rejection_layer = unknown_rejection_layer
        self.center_loss_weight      = center_loss_weight
        self.unknown_loss_weight     = unknown_loss_weight
        self.physics_warmup_epochs   = physics_warmup_epochs
        self.current_epoch           = tf.Variable(0, trainable=False, dtype=tf.int32)

        # Metrics
        self._loss_names = [
            "loss", "arcface_loss", "center_loss", "unknown_loss",
            "sym_loss", "energy_loss", "lap_loss",
            "accuracy", "embed_norm",
            "energy_known", "energy_unknown", "energy_sep"
        ]
        self._trackers = {
            n: keras.metrics.Mean(name=n) for n in self._loss_names
            if n != "accuracy"
        }
        self._trackers["accuracy"] = keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )

    def call(self, inputs, training=False):
        return self.base_model(inputs, training=training)

    @property
    def metrics(self):
        return list(self._trackers.values())

    def _reset_trackers(self):
        for t in self._trackers.values():
            t.reset_state()

    # Training step
    def train_step(self, data):
        inputs, labels = data
        face_input     = inputs['face_input']
        labels         = tf.cast(labels, tf.int32)

        λ_sym, λ_energy, λ_lap = get_physics_weights_tf(
            self.current_epoch, self.physics_warmup_epochs
        )

        with tf.GradientTape() as tape:
            logits, embeddings, embeddings_unnorm = self.base_model(
                inputs, training=True
            )

            known_mask          = tf.greater_equal(labels, 0)
            unknown_mask        = tf.less(labels, 0)
            known_emb           = tf.boolean_mask(embeddings,        known_mask)
            known_emb_unnorm    = tf.boolean_mask(embeddings_unnorm, known_mask)

            # L_arc
            def arcface_loss_fn():
                kl = tf.boolean_mask(logits, known_mask)
                ky = tf.boolean_mask(labels, known_mask)
                return tf.reduce_mean(
                    tf.nn.sparse_softmax_cross_entropy_with_logits(
                        labels=ky, logits=kl
                    )
                )
            arcface_loss = tf.cond(
                tf.reduce_any(known_mask),
                arcface_loss_fn,
                lambda: tf.constant(0.0)
            )

            # L_center 
            center_loss = self.center_loss_layer(embeddings, labels, training=True)

            # L_unknown
            arcface_W   = self.base_model.get_layer('arcface').W
            unknown_loss = self.unknown_rejection_layer(embeddings, labels, arcface_W)

            # L_sym + L_lap
            def physics_sym_lap():
                known_face = tf.boolean_mask(face_input, known_mask)

                # stop_gradient on augmented inputs — no gradient through them
                flipped   = tf.stop_gradient(tf.image.flip_left_right(known_face))
                noise     = tf.random.normal(tf.shape(known_face), stddev=0.1)
                perturbed = tf.stop_gradient(known_face + noise)

                emb_flip, _ = self.embedding_model(flipped,   training=True)
                emb_aug,  _ = self.embedding_model(perturbed, training=True)

                sym = tf.reduce_mean(
                    tf.reduce_sum(tf.square(known_emb - emb_flip), axis=1)
                )
                lap = tf.reduce_mean(
                    tf.reduce_sum(tf.square(known_emb - emb_aug),  axis=1)
                )
                return sym, lap

            sym_loss, lap_loss = tf.cond(
                tf.reduce_any(known_mask),
                physics_sym_lap,
                lambda: (tf.constant(0.0), tf.constant(0.0))
            )

            # L_energy 
            def energy_loss_fn():
                unk_emb_unnorm = tf.boolean_mask(embeddings_unnorm, unknown_mask)
                loss, E_k, E_u = energy_margin_loss(
                    known_emb_unnorm, unk_emb_unnorm, arcface_W,
                    margin=CFG.ENERGY_MARGIN, tau=CFG.ENERGY_TAU
                )
                self._trackers["energy_known"].update_state(E_k)
                self._trackers["energy_unknown"].update_state(E_u)
                self._trackers["energy_sep"].update_state(E_u - E_k)
                return loss

            has_both   = tf.logical_and(
                tf.reduce_any(known_mask), tf.reduce_any(unknown_mask)
            )
            energy_loss = tf.cond(has_both, energy_loss_fn,
                                  lambda: tf.constant(0.0))

            # Total loss 
            total_loss = (
                arcface_loss
                + self.center_loss_weight  * center_loss
                + self.unknown_loss_weight * unknown_loss
                + λ_sym    * sym_loss
                + λ_energy * energy_loss
                + λ_lap    * lap_loss
            )
            if self.base_model.losses:
                total_loss += tf.add_n(self.base_model.losses)

        grads = tape.gradient(total_loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(
            zip(grads, self.base_model.trainable_variables)
        )

        # Update trackers 
        self._trackers["loss"].update_state(total_loss)
        self._trackers["arcface_loss"].update_state(arcface_loss)
        self._trackers["center_loss"].update_state(center_loss)
        self._trackers["unknown_loss"].update_state(unknown_loss)
        self._trackers["sym_loss"].update_state(sym_loss)
        self._trackers["energy_loss"].update_state(energy_loss)
        self._trackers["lap_loss"].update_state(lap_loss)
        self._trackers["embed_norm"].update_state(
            tf.reduce_mean(tf.norm(embeddings, axis=1))
        )
        def update_acc():
            self._trackers["accuracy"].update_state(
                tf.boolean_mask(labels, known_mask),
                tf.boolean_mask(logits, known_mask)
            )
            return tf.constant(0)
        tf.cond(tf.reduce_any(known_mask), update_acc, lambda: tf.constant(0))

        return {n: self._trackers[n].result() for n in self._loss_names}

    # Validation step 
    def test_step(self, data):
        inputs, labels = data
        labels         = tf.cast(labels, tf.int32)

        logits, embeddings, _ = self.base_model(inputs, training=False)
        known_mask = tf.greater_equal(labels, 0)

        def val_arcface():
            return tf.reduce_mean(
                tf.nn.sparse_softmax_cross_entropy_with_logits(
                    labels=tf.boolean_mask(labels, known_mask),
                    logits=tf.boolean_mask(logits, known_mask)
                )
            )
        arcface_loss = tf.cond(
            tf.reduce_any(known_mask), val_arcface, lambda: tf.constant(0.0)
        )
        center_loss  = self.center_loss_layer(embeddings, labels, training=False)
        total_loss   = arcface_loss + self.center_loss_weight * center_loss
        if self.base_model.losses:
            total_loss += tf.add_n(self.base_model.losses)

        self._trackers["loss"].update_state(total_loss)
        self._trackers["arcface_loss"].update_state(arcface_loss)
        self._trackers["center_loss"].update_state(center_loss)
        self._trackers["embed_norm"].update_state(
            tf.reduce_mean(tf.norm(embeddings, axis=1))
        )
        def update_val_acc():
            self._trackers["accuracy"].update_state(
                tf.boolean_mask(labels, known_mask),
                tf.boolean_mask(logits, known_mask)
            )
            return tf.constant(0)
        tf.cond(tf.reduce_any(known_mask), update_val_acc, lambda: tf.constant(0))

        return {
            "loss": self._trackers["loss"].result(),
            "arcface_loss": self._trackers["arcface_loss"].result(),
            "center_loss": self._trackers["center_loss"].result(),
            "accuracy": self._trackers["accuracy"].result(),
            "embed_norm": self._trackers["embed_norm"].result(),
        }


class EpochCounterCallback(keras.callbacks.Callback):
    """Keeps trainer.current_epoch in sync and prints physics weights."""
    def __init__(self, trainer):
        super().__init__()
        self.trainer = trainer

    def on_epoch_begin(self, epoch, logs=None):
        self.trainer.current_epoch.assign(epoch)
        λs, λe, λl = get_physics_weights_tf(
            epoch, self.trainer.physics_warmup_epochs
        )
        print(f"\n  [Epoch {epoch+1}] "
              f"λ_sym={λs:.5f}  λ_energy={λe:.5f}  λ_lap={λl:.5f}")

class LossVerificationCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0:
            print("\n  ── LOSS VERIFICATION (Epoch 1) ──")
            for k, v in logs.items():
                print(f"    {k}: {v:.6f}")
            print("  ── If any loss that should be active shows 0.000000, STOP. ──")



# CURRICULUM MIXING


def make_curriculum_dataset(known_ds, unknown_ratio):
    """
    If unknown_ratio == 0: return pure known dataset.
    Otherwise: slice each batch into known + heavily-augmented synthetic unknowns.
    Unknowns get label = -1.
    """
    if unknown_ratio == 0.0:
        return known_ds.batch(CFG.BATCH_SIZE).prefetch(CFG.AUTOTUNE)

    n_known   = max(4, int(CFG.BATCH_SIZE * (1.0 - unknown_ratio)))
    n_unknown = CFG.BATCH_SIZE - n_known
    print(f"  Batch mix: {n_known} known + {n_unknown} synthetic unknown")

    def create_mixed_batch(inputs, labels):
        imgs = inputs['face_input']

        known_imgs   = imgs[:n_known]
        known_labels = labels[:n_known]

        unk_source = imgs[n_known:n_known + n_unknown]
        unk_imgs   = tf.map_fn(augment_unknown, unk_source, dtype=tf.float32)
        unk_labels = tf.fill([n_unknown], -1)

        all_imgs   = tf.concat([known_imgs, unk_imgs],   axis=0)
        all_labels = tf.concat([known_labels, unk_labels], axis=0)

        # Shuffle within batch
        idx        = tf.random.shuffle(tf.range(tf.shape(all_labels)[0]))
        all_imgs   = tf.gather(all_imgs,   idx)
        all_labels = tf.gather(all_labels, idx)

        merged_inputs = {'face_input': all_imgs, 'label_input': all_labels}
        return merged_inputs, all_labels

    return (known_ds
            .batch(n_known + n_unknown)
            .map(create_mixed_batch, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))



# BUILD TRAINER


trainer = PhysicsTrainer(
    base_model=model,
    embedding_model=embedding_model,
    center_loss_layer=center_loss_layer,
    unknown_rejection_layer=unknown_rejection_layer,
    center_loss_weight=CFG.CENTER_LOSS_WEIGHT,
    unknown_loss_weight=CFG.UNKNOWN_LOSS_WEIGHT,
    physics_warmup_epochs=CFG.PHYSICS_WARMUP_EPOCHS
)

# Validation dataset (no augmentation, no shuffle, no repeat)
val_ds = (val_data.get_tfdata_unbatched(shuffle=False, augment=False)
          .batch(CFG.BATCH_SIZE)
          .prefetch(CFG.AUTOTUNE))
val_steps = int(np.ceil(val_data.total_images / CFG.BATCH_SIZE))

# Warm up model variables with a dummy forward pass
print("\nInitialising model variables...")
dummy = next(iter(val_ds))
_ = trainer(dummy[0], training=False)
print("  ✓ Variables created")

trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))



# TRAINING — THREE PHASES


print(f"\n{'='*70}")
print(f"  PHASE 1 — Warmup ({CFG.PHASE_1_EPOCHS} epochs, 100% known)")
print(f"{'='*70}")

train_ds_p1 = make_curriculum_dataset(
    train_data.get_tfdata_unbatched(shuffle=True, augment=True),
    unknown_ratio=0.0
)

history_p1 = trainer.fit(
    train_ds_p1,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=val_steps,
    epochs=CFG.PHASE_1_EPOCHS,
    callbacks=[
        EpochCounterCallback(trainer),
        LossVerificationCallback(),
        keras.callbacks.ModelCheckpoint(
            CFG.ckpt(1), monitor='val_loss',
            save_best_only=True, save_weights_only=True, verbose=1
        ),
    ]
)

# Phase 2 
print(f"\n{'='*70}")
print(f"  PHASE 2 — Introduction ({CFG.PHASE_2_EPOCHS} epochs, 80/20 known/unknown)")
print(f"{'='*70}")

train_ds_p2 = make_curriculum_dataset(
    train_data.get_tfdata_unbatched(shuffle=True, augment=True),
    unknown_ratio=0.2
)
trainer.load_weights(CFG.ckpt(1))

history_p2 = trainer.fit(
    train_ds_p2,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=val_steps,
    initial_epoch=CFG.PHASE_1_EPOCHS,
    epochs=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    callbacks=[
        EpochCounterCallback(trainer),
        LossVerificationCallback(),
        keras.callbacks.ModelCheckpoint(
            CFG.ckpt(2), monitor='val_loss',
            save_best_only=True, save_weights_only=True, verbose=1
        ),
    ]
)

# Phase 3 
print(f"\n{'='*70}")
print(f"  PHASE 3 — Hardening ({CFG.PHASE_3_EPOCHS} epochs, 60/40 known/unknown)")
print(f"{'='*70}")

train_ds_p3 = make_curriculum_dataset(
    train_data.get_tfdata_unbatched(shuffle=True, augment=True),
    unknown_ratio=0.4
)
trainer.load_weights(CFG.ckpt(2))
trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4))  # lower LR

history_p3 = trainer.fit(
    train_ds_p3,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=val_steps,
    initial_epoch=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    epochs=CFG.TOTAL_EPOCHS,
    callbacks=[
        EpochCounterCallback(trainer),
        LossVerificationCallback(),
        keras.callbacks.ModelCheckpoint(
            CFG.ckpt("final"), monitor='val_loss',
            save_best_only=True, save_weights_only=True, verbose=1
        ),
    ]
)

print("\n✓ TRAINING COMPLETE")



# EXPORT — Inference model + Training gallery + Class centres + Metadata


print("\nExporting...")
trainer.load_weights(CFG.ckpt("final"))

# 1. SavedModel (normalized embeddings only) 
inference_model = keras.Model(
    inputs=trainer.base_model.input[0],
    outputs=trainer.base_model.output[1],   # L2-normalized embeddings
    name=f"cow_{CFG.MODEL_TAG}_inference"
)
tf.saved_model.save(inference_model, CFG.saved_model_dir())
print(f"  ✓ Saved model  → {CFG.saved_model_dir()}/")

# 2. ArcFace class centres 
arcface_W     = model.get_layer('arcface').W.numpy()
arcface_W_norm = arcface_W / (np.linalg.norm(arcface_W, axis=0, keepdims=True) + 1e-8)
np.save(CFG.centers_path(), arcface_W_norm.T)   # shape: (num_classes, embed_dim)
print(f"  ✓ Class centres → {CFG.centers_path()}")

# 3. Training gallery (embeddings for ALL training images) 
#
#  This is the key addition for genuine-score evaluation.
#  The evaluation script will:
#    - Use these embeddings as the "known" gallery
#    - Score external test images against them to get genuine/impostor scores
#
print(f"\nExtracting training gallery embeddings...")
train_paths, train_labels, train_classes = train_data.get_all_paths_and_labels()

GALLERY_BATCH = 64
all_embeddings = []

for i in range(0, len(train_paths), GALLERY_BATCH):
    batch_paths = train_paths[i : i + GALLERY_BATCH]
    imgs = tf.stack([load_image(p) for p in batch_paths], axis=0)
    embs = inference_model(imgs, training=False).numpy()
    all_embeddings.append(embs)
    if (i // GALLERY_BATCH) % 10 == 0:
        print(f"  {min(i + GALLERY_BATCH, len(train_paths))}/{len(train_paths)} images processed")

all_embeddings = np.concatenate(all_embeddings, axis=0)

# Per-cattle mean embedding (the actual gallery vector used at query time)
labels_np       = np.array(train_labels)
mean_embeddings = np.zeros((train_data.num_classes, CFG.EMBED_DIM), dtype=np.float32)
for cls_idx in range(train_data.num_classes):
    mask = labels_np == cls_idx
    if mask.sum() > 0:
        vecs = all_embeddings[mask]
        mean = vecs.mean(axis=0)
        mean_embeddings[cls_idx] = mean / (np.linalg.norm(mean) + 1e-8)

np.savez(
    CFG.gallery_path(),
    embeddings=all_embeddings,          # per-image embeddings
    mean_embeddings=mean_embeddings,    # per-cattle mean (primary gallery)
    labels=labels_np,
    class_names=np.array(train_classes)
)
print(f"  ✓ Training gallery → {CFG.gallery_path()}")
print(f"    {len(train_paths)} images  |  {train_data.num_classes} cattle  "
      f"|  embedding dim = {CFG.EMBED_DIM}")

# 4. Metadata 
metadata = {
    'model_tag':          CFG.MODEL_TAG,
    'architecture':       'EfficientNetB0 → Dense(512) → L2-Norm → ArcFace',
    'training_cattle':    train_data.num_classes,
    'training_images':    train_data.total_images,
    'data_source':        'face_train + face_val + face_test (combined)',
    'train_val_split':    f'{CFG.TRAIN_SPLIT*100:.0f}/{(1-CFG.TRAIN_SPLIT)*100:.0f}',
    'loss_weights': {
        'center':   CFG.CENTER_LOSS_WEIGHT,
        'unknown':  CFG.UNKNOWN_LOSS_WEIGHT,
        'sym':      CFG.PHYSICS_SYM_WEIGHT,
        'lap':      CFG.PHYSICS_LAP_WEIGHT,
        'energy':   CFG.PHYSICS_ENERGY_WEIGHT,
    },
    'physics': {
        'warmup_epochs': CFG.PHYSICS_WARMUP_EPOCHS,
        'energy_tau':    CFG.ENERGY_TAU,
        'energy_margin': CFG.ENERGY_MARGIN,
    },
    'training': {
        'phase1_epochs':    CFG.PHASE_1_EPOCHS,
        'phase2_epochs':    CFG.PHASE_2_EPOCHS,
        'phase3_epochs':    CFG.PHASE_3_EPOCHS,
        'steps_per_epoch':  CFG.STEPS_PER_EPOCH,
        'batch_size':       CFG.BATCH_SIZE,
    },
    'outputs': {
        'saved_model':       CFG.saved_model_dir(),
        'class_centres':     CFG.centers_path(),
        'training_gallery':  CFG.gallery_path(),
    }
}
with open(CFG.metadata_path(), 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"  ✓ Metadata        → {CFG.metadata_path()}")

print(f"\n{'='*70}")
print(f"  {CFG.MODEL_TAG} COMPLETE — ready for evaluation")
print(f"  Gallery : {CFG.gallery_path()}")
print(f"  Model   : {CFG.saved_model_dir()}/")
print(f"{'='*70}")

2026-04-21 16:39:59.336380: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-21 16:39:59.468522: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-21 16:39:59.510326: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-21 16:40:00.205793: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; 

  TRAINING  L3
  Weights → unknown=0.3  sym=0.05  lap=0.05  energy=0.0

LOADING DATA

✓ 468 cattle loaded
  Train : 3774 images (85%)
  Val   : 952   images (15%)

BUILDING MODEL


2026-04-21 16:40:01.257482: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-21 16:40:01.908154: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14492 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:d8:00.0, compute capability: 7.5


  ✓ Model built: EfficientNetB0 → Dense(512) → L2-Norm → ArcFace
  ✓ Embedding extractor ready

Initialising model variables...


2026-04-21 16:40:05.418406: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8201


  ✓ Variables created

  PHASE 1 — Warmup (5 epochs, 100% known)

  [Epoch 1] λ_sym=0.00000  λ_energy=0.00000  λ_lap=0.00000
Epoch 1/5
200/200 [==============================] - ETA: 0s - loss: 25.3952 - arcface_loss: 25.2187 - center_loss: 0.8200 - unknown_loss: 0.0000e+00 - sym_loss: 0.7704 - energy_loss: 0.0000e+00 - lap_loss: 0.7099 - accuracy: 0.0256 - embed_norm: 1.0000 - energy_known: 0.0000e+00 - energy_unknown: 0.0000e+00 - energy_sep: 0.0000e+00
  ── LOSS VERIFICATION (Epoch 1) ──
    loss: 25.395210
    arcface_loss: 25.218691
    center_loss: 0.820011
    unknown_loss: 0.000000
    sym_loss: 0.770378
    energy_loss: 0.000000
    lap_loss: 0.709888
    accuracy: 0.025625
    embed_norm: 1.000000
    energy_known: 0.000000
    energy_unknown: 0.000000
    energy_sep: 0.000000
    val_loss: 11.199304
    val_arcface_loss: 11.013398
    val_center_loss: 0.682011
    val_accuracy: 0.215625
    val_embed_norm: 1.000000
  ── If any loss that should be active shows 0.000000, STOP.

INFO:tensorflow:Assets written to: cow_L3_saved_model/assets


INFO:tensorflow:Assets written to: cow_L3_saved_model/assets


  ✓ Saved model  → cow_L3_saved_model/
  ✓ Class centres → cow_L3_class_centers.npy

Extracting training gallery embeddings...
  64/3774 images processed
  704/3774 images processed
  1344/3774 images processed
  1984/3774 images processed
  2624/3774 images processed
  3264/3774 images processed
  ✓ Training gallery → cow_L3_train_gallery.npz
    3774 images  |  468 cattle  |  embedding dim = 512
  ✓ Metadata        → cow_L3_metadata.json

  L3 COMPLETE — ready for evaluation
  Gallery : cow_L3_train_gallery.npz
  Model   : cow_L3_saved_model/
